# Introduction


**File:** Notebook_to_build_graphics.ipynb 
\
**Author:** Elizabeth Gould
\
**Date:** 08.05.2026
\
**Problem:** Test our model for $M_{\mu}\left(BR, dk, \mu, \psi_0^\prime\right)$

This code contains a Monte Carlo test of our model for $M_{\mu}\left(BR, dk, \mu, \psi_0^\prime\right)$.

# Test of Slow Oscillation Wavenumber Model

This is code to visually and numerically test the fit of my slow oscillation model. The model is not perfect, but good enough.

In [ ]:
import eelib
import numpy as np
import matplotlib.pyplot as plt
import sklearn.metrics as mtc

In [ ]:
#--PARAMETERS--------

# note that k, B, and R are percents here, mu is not as its scale is unknown
dk = 0.5
R  = 1.0
B  = 0.5
mu = 1.0e-6

n_mc = 1000
n_graph = 1000

b_r  = (0.5, 1.0)  # B here is B*R
mu_r = (1.0e-8, 1.0e-6)    # 1.0e-8 to 1.0e-6
k_r = (0.1,0.9)


In [4]:
#--CODE--

# Make the grid object
gridl = eelib.grid_slow_osc(R, B, dk, mu)

# Save our full solution
gridl.save_solution = True

# Make the grid
gridl.makeMCPoints(mu=mu_r, dk=k_r, B=b_r, num = n_mc)

# Run the grid
gridl.mcSlowOsc()

Begin grid build:  0.0
Number of periods to calculate: 1000
Done grid build:  44238.497473955154


In [13]:
res = []
for i in range(n_mc):
    res.append(gridl.slow_osc_sol[i]['y'][0][-1])

In [ ]:
# Now for the rest of the numbers
max_arr = np.zeros((n_mc,3))
ave_arr = np.zeros((n_mc,3))
std_arr = np.zeros((n_mc,3))
rmse_arr = np.zeros((n_mc,3))
mape_arr = np.zeros((n_mc,3))
r2_arr = np.zeros((n_mc,3))
arr_wn = np.zeros((n_mc,2))
arr_fp = np.zeros((n_mc,2), dtype="complex")
arr_max = np.zeros(n_mc)
for ii in range(n_mc):
    # First, declare which solution I am using
    sol = gridl.slow_osc_sol[ii]

    # Retrieve my fits for this function from the grid.
    MM    = gridl.slow_osc_k[ii]
    amp   = gridl.slow_osc_a[ii]
    theta = gridl.slow_osc_th[ii]

    # Predicted slow oscillation amplitude.
    vt = gridl.val_table[ii] # mu, dk, B, R, A, k0, dr, di
    MM2 = eelib.pred_slow_k(vt[6]+1j*vt[7], vt[0], vt[0], vt[2]*eelib.B_max, vt[3]*eelib.R_max, vt[4], vt[5]) #dpsi0, mu, dk, B, R, A = 1., k0=kFAu

    arr_wn[ii] = [MM, MM2]

    # Pull the found solution from the sol variable.
    t_list  = sol['t'][:-1]
    y1_list = np.real(sol['y'][0][:-1])

    # Estimated fit
    y2_list = amp * np.sin(MM2 * (t_list) + theta)

    # Fit to a sin using SciPy
    y3_list = amp * np.sin(MM * t_list + theta)

    y_end = sol['y'][0][-1]
    arr_fp[ii] = [y_end, gridl.slow_osc_sol_1[ii]]
    arr_max[ii] = np.max(np.abs(y1_list))

    max_arr[ii, :] = [np.max(np.abs(y3_list-y2_list)), np.max(np.abs(y1_list-y2_list)), np.max(np.abs(y1_list-y3_list))]
    ave_arr[ii, :] = [np.average(y3_list-y2_list), np.average(y1_list-y2_list), np.average(y1_list-y3_list)]
    std_arr[ii, :] = [np.std(y3_list-y2_list), np.std(y1_list-y2_list), np.std(y1_list-y3_list)]
    rmse_arr[ii, :] = [mtc.root_mean_squared_error(y3_list,y2_list)/np.max(np.abs(y1_list)), mtc.root_mean_squared_error(y1_list,y2_list)/np.max(np.abs(y1_list)), mtc.root_mean_squared_error(y1_list,y3_list)/np.max(np.abs(y1_list))]
    mape_arr[ii, :] = [mtc.mean_absolute_percentage_error(y3_list,y2_list), mtc.mean_absolute_percentage_error(y1_list,y2_list), mtc.mean_absolute_percentage_error(y1_list,y3_list)]
    r2_arr[ii, :] = [mtc.r2_score(y3_list,y2_list), mtc.r2_score(y1_list,y2_list), mtc.r2_score(y1_list,y3_list)]

In [ ]:
print("curve rel rmse", np.sqrt(np.average(np.power(rmse_arr[:,1]/arr_max, 2))))
print("curve rmse", np.sqrt(np.average(np.power(rmse_arr[:,1], 2))))
print("m rmse", mtc.root_mean_squared_error(arr_wn[:,0], arr_wn[:,1]))
print("m rel rmse", np.sqrt(np.average(np.power((arr_wn[:,0]-arr_wn[:,1])/arr_wn[:,0], 2))))
print("fin point rmse", mtc.root_mean_squared_error(arr_fp[:,0],arr_fp[:,1]))
print("fin point rel rmse", np.sqrt(np.sum(np.power(np.abs(arr_fp[:,0]-arr_fp[:,1]), 2))))

In [ ]:
print("curve min r2", np.min(r2_arr[:,1]))
print("curve ave r2", np.average(r2_arr[:,1]))
print("m r2", mtc.r2_score(arr_wn[:,0], arr_wn[:,1]))

In [ ]:
print("curve max", np.max(max_arr[:,1]))
print("curve rel max", np.max(max_arr[:,1]/arr_max))
print("m max", np.max(np.abs((arr_wn[:,0]-arr_wn[:,1]))))
print("m rel max", np.max(np.abs((arr_wn[:,0]-arr_wn[:,1])/arr_wn[:,0])))
print("fin max", np.max(np.abs(arr_fp[:,0]-arr_fp[:,1])))
print("fin rel max", np.max(np.abs(arr_fp[:,0]-arr_fp[:,1])))

In [ ]:
print("curve mean", np.ave(ave_arr[:,1]))
print("curve std", np.sqrt((n_mc*n_graph)/(n_mc*n_graph-1))*np.sqrt(np.average(np.power(std_arr[:,1],2))))
print("m mean", np.average(arr_wn[:,0]-arr_wn[:,1]))
print("m std", np.sqrt(n_mc/(n_mc-1))*np.std(arr_wn[:,0]-arr_wn[:,1]))
print("fin mean", np.average(arr_fp[:,0]-arr_fp[:,1]))
print("fin std", np.sqrt(n_mc/(n_mc-1))*np.std(arr_fp[:,0]-arr_fp[:,1]))

In [ ]:
print("curve mape", np.average(mape_arr[:,1]))
print("m mape", mtc.mean_absolute_percentage_error(arr_wn[:,0], arr_wn[:,1]))
print("fin mape", mtc.mean_absolute_percentage_error(arr_fp[:,0],arr_fp[:,1]))